In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 02: Data Augmentation

**Paper stage:** Data pre-processing layer — Data augmentation (paper Section 3)

**Order (paper Fig. 1 / Fig. 5):** augmentation happens on the WHOLE dataset **before** imputation, wrangling, versioning and the split.

| Group | Transforms | Variants |
|-------|-----------|----------|
| **A — paper** | rotations 0/30/45/90/180/270 (0° = original); the same 6 rotations of the horizontally flipped image | 12 |
| **B — additions** | translation ±15% of side (≈34 px) L/R/U/D; zoom-in 1.25× (centre crop 80%); zoom-out 0.8× (on black canvas); 4 corner crops of 75% | 10 |
| | **Total per original image** | **22** |

All transforms: deterministic (no randomness), applied to the **original** image only (one transform per output, no stacking), 224×224 RGB after a plain resize, black fill, bilinear interpolation, no canvas expansion. Vertical flip is NOT added (= hflip + 180°).

## Common setup

Same first cell in every notebook: mount Drive, create the project folders, seed 42, print library versions and GPU.

In [2]:
# ===== COMMON SETUP CELL (same in every notebook, spec section 4) =====
import os, sys, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# [kaggle] Colab Drive mount removed - bsm_env recreates the folder tree

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
for sub in ["data", "augmented", "arrays", "checkpoints", "results", "figures", "versions"]:
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("project root :", PROJECT)
print("python       :", sys.version.split()[0])
print("numpy        :", np.__version__)
print("torch        :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
try:
    import torchvision; print("torchvision  :", torchvision.__version__)
except Exception:
    pass
print("setup OK")

project root : /content/drive/MyDrive/bsm_paper_replication
python       : 3.13.15
numpy        : 2.1.3
torch        : 2.11.0+cu128
cuda available: True
GPU          : Tesla T4
torchvision  : 0.26.0+cu128
setup OK


## Variant table (configurable) and transform definitions

The full list of the 22 variants lives in `VARIANT_TABLE` at the top. All transforms map the resized 224×224 original to one output.

In [3]:
# ===== Paper stage: augmentation — variant table + transforms =====
import os, time, shutil
import pandas as pd
from PIL import Image

IMG_SIZE = 224
CLASS_ORDER = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]

PAPER_ROTATIONS = [0, 30, 45, 90, 180, 270]
SHIFT_FRAC = 0.15
ZOOM_IN = 1.25
ZOOM_OUT = 0.8
CROP_FRAC = 0.75

VARIANT_TABLE = []
def _add(group, aug_type, name, params):
    VARIANT_TABLE.append({"aug_group": group, "aug_type": aug_type,
                          "variant_name": name, "parameters": params})

for deg in PAPER_ROTATIONS:
    _add("paper", "original" if deg == 0 else "rotation", f"rot{deg}", f"rotation={deg}deg")
for deg in PAPER_ROTATIONS:
    _add("paper", "hflip_rotation", f"hflip_rot{deg}", f"hflip=1,rotation={deg}deg")
for name, dx, dy in [("left", -1, 0), ("right", 1, 0), ("up", 0, -1), ("down", 0, 1)]:
    _add("addition", "translation", f"shift_{name}", f"dx={dx},dy={dy},frac={SHIFT_FRAC}")
_add("addition", "zoom", "zoom_in_1.25", f"scale={ZOOM_IN},centre_crop_frac={1/ZOOM_IN:.2f}")
_add("addition", "zoom", "zoom_out_0.80", f"scale={ZOOM_OUT},canvas=black")
for corner, box in [("tl", (0, 0)), ("tr", (1, 0)), ("bl", (0, 1)), ("br", (1, 1))]:
    _add("addition", "crop", f"crop_{corner}", f"frac={CROP_FRAC},corner={corner}")

vt = pd.DataFrame(VARIANT_TABLE)
print(f"variants per image: {len(vt)}  (paper:{(vt.aug_group=='paper').sum()}, add:{(vt.aug_group=='addition').sum()})")
assert len(vt) == 22
print(vt.to_string(index=False))

def t_original(im): return im.copy()
def _rot(im, deg): return im.rotate(deg, resample=Image.BILINEAR, expand=False, fillcolor=(0,0,0))
def _hflip(im): return im.transpose(Image.FLIP_LEFT_RIGHT)

def apply_variant(im, row):
    name, atype = row["variant_name"], row["aug_type"]
    if atype == "original": return t_original(im)
    if atype == "rotation": return _rot(im, int(name.replace("rot","")))
    if atype == "hflip_rotation":
        deg = int(name.replace("hflip_rot",""))
        return _rot(_hflip(im), deg)
    if atype == "translation":
        frac = SHIFT_FRAC
        dx = round(frac*IMG_SIZE)*(1 if "right" in name else -1 if "left" in name else 0)
        dy = round(frac*IMG_SIZE)*(1 if "down" in name else -1 if "up" in name else 0)
        return im.transform(im.size, Image.AFFINE, (1,0,dx,0,1,dy), resample=Image.BILINEAR, fillcolor=(0,0,0))
    if atype == "zoom" and name == "zoom_in_1.25":
        side = round(IMG_SIZE/ZOOM_IN); o = (IMG_SIZE-side)/2
        return im.crop((o,o,o+side,o+side)).resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR)
    if atype == "zoom" and name == "zoom_out_0.80":
        small = im.resize((round(IMG_SIZE*ZOOM_OUT),)*2, Image.BILINEAR)
        canvas = Image.new("RGB",(IMG_SIZE,IMG_SIZE),(0,0,0))
        o = (IMG_SIZE-small.size[0])//2; canvas.paste(small,(o,o)); return canvas
    if atype == "crop":
        side = round(IMG_SIZE*CROP_FRAC)
        x0 = 0 if name.endswith(("tl","bl")) else IMG_SIZE-side
        y0 = 0 if name.endswith(("tl","tr")) else IMG_SIZE-side
        return im.crop((x0,y0,x0+side,y0+side)).resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR)
    raise ValueError
print("transforms loaded OK")

variants per image: 22  (paper:12, add:10)
aug_group       aug_type  variant_name                       parameters
    paper       original          rot0                    rotation=0deg
    paper       rotation         rot30                   rotation=30deg
    paper       rotation         rot45                   rotation=45deg
    paper       rotation         rot90                   rotation=90deg
    paper       rotation        rot180                  rotation=180deg
    paper       rotation        rot270                  rotation=270deg
    paper hflip_rotation    hflip_rot0            hflip=1,rotation=0deg
    paper hflip_rotation   hflip_rot30           hflip=1,rotation=30deg
    paper hflip_rotation   hflip_rot45           hflip=1,rotation=45deg
    paper hflip_rotation   hflip_rot90           hflip=1,rotation=90deg
    paper hflip_rotation  hflip_rot180          hflip=1,rotation=180deg
    paper hflip_rotation  hflip_rot270          hflip=1,rotation=270deg
 addition    translat

## Generate the augmented dataset (resumable)

VM already has local files from a previous run in this session; this cell will RESUME instantly (8,074 PNGs). After generation/resume, syncs the local tree to Drive `augmented/<class>/` (skip existing files).

In [4]:
# ===== Generate (resume) + sync to Drive =====
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
LOCAL_ROOT = "/content/aug_local"
LOCAL_MANIFEST = "/content/augmentation_manifest.csv"

inv = pd.read_csv(os.path.join(PROJECT, "results", "raw_inventory.csv"))
print("originals:", len(inv))

rows = None
if os.path.exists(LOCAL_MANIFEST):
    old = pd.read_csv(LOCAL_MANIFEST)
    if len(old) == len(inv) * len(VARIANT_TABLE):
        rows = old.to_dict("records")
        print(f"RESUMED: manifest complete ({len(rows)} rows) — skipping local generation")
if rows is None:
    rows, skipped = [], []
    t0 = time.time()
    for n, rec in inv.iterrows():
        try:
            with Image.open(rec["filepath"]) as im:
                im = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                for v in VARIANT_TABLE:
                    out = apply_variant(im, v)
                    cls_dir = os.path.join(LOCAL_ROOT, rec["class_name"])
                    os.makedirs(cls_dir, exist_ok=True)
                    aug_id = f"{rec['class_name']}_{rec['image_id']}_{v['variant_name']}"
                    fpath = os.path.join(cls_dir, aug_id + ".png")
                    out.save(fpath, format="PNG")
                    rows.append({
                        "aug_id": aug_id,
                        "source_image_id": f"{rec['class_name']}_{rec['image_id']}",
                        "class_name": rec["class_name"], "class_id": int(rec["class_id"]),
                        "aug_group": v["aug_group"], "aug_type": v["aug_type"],
                        "variant_name": v["variant_name"], "parameters": v["parameters"],
                        "filepath": fpath
                    })
        except Exception as e:
            skipped.append({"image_id": rec["image_id"], "class_name": rec["class_name"], "reason": f"{type(e).__name__}:{e}"})
        if (n+1) % 50 == 0:
            el = time.time()-t0; rate=(n+1)/el; eta=(len(inv)-n-1)/rate
            print(f"{n+1}/{len(inv)} {el:.0f}s eta {eta:.0f}s {len(rows)} variants")
    pd.DataFrame(rows).to_csv(LOCAL_MANIFEST, index=False)
    pd.DataFrame(skipped, columns=["image_id","class_name","reason"]).to_csv(os.path.join(PROJECT,"results","augmentation_skipped.csv"), index=False)
    print(f"generated {len(rows)} PNGs in {time.time()-t0:.0f}s skipped {len(skipped)}")

m = pd.DataFrame(rows)
print("manifest rows:", len(m), "unique aug_id:", m["aug_id"].is_unique)
print(m.groupby(["aug_group","aug_type"]).size().to_string())

# Sync to Drive: copy local tree to PROJECT/augmented (skip existing)
t0 = time.time()
drive_root = os.path.join(PROJECT, "augmented")
copied = skipped_f = 0
for cls in m["class_name"].unique():
    src = os.path.join(LOCAL_ROOT, cls); dst = os.path.join(drive_root, cls)
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        sp, dp = os.path.join(src,f), os.path.join(dst,f)
        if os.path.exists(dp):
            skipped_f += 1
            continue
        shutil.copy2(sp, dp); copied += 1
el = time.time()-t0
print(f"DRIVE SYNC: copied {copied} files ({skipped_f} existed) in {el:.1f}s")


originals: 367
50/367 15s eta 94s 1100 variants
100/367 31s eta 82s 2200 variants
150/367 47s eta 68s 3300 variants
200/367 61s eta 51s 4400 variants
250/367 75s eta 35s 5500 variants
300/367 93s eta 21s 6600 variants
350/367 110s eta 5s 7700 variants
generated 8074 PNGs in 116s skipped 0
manifest rows: 8074 unique aug_id: True
aug_group  aug_type      
addition   crop              1468
           translation       1468
           zoom               734
paper      hflip_rotation    2202
           original           367
           rotation          1835
DRIVE SYNC: copied 8074 files (0 existed) in 1.3s


In [5]:
import os, glob
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
aug = glob.glob(os.path.join(PROJECT, "augmented", "*", "*.png"))
print("augmented png count on Drive:", len(aug))


augmented png count on Drive: 8074


## Summary: augmented dataset counts vs paper

Print actual counts (8074) and compare to paper's 1468 (from its stated 12-variant transform in the description). Never force 1468.

In [6]:
# ===== Augmentation summary (paper comparison) =====
import os, pandas as pd

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
m = pd.read_csv("/content/augmentation_manifest.csv")
summ = {
  "original_count": len(m["source_image_id"].unique()),
  "variants_per_image": int(len(m) / len(m["source_image_id"].unique())),
  "augmented_count": len(m),
  "factor": int(len(m) / len(m["source_image_id"].unique())),
}
print("SUMMARY (ACTUAL)")
for k,v in summ.items():
    print(f"{k:20s}: {v}")
print("\nPAPER (for comparison only): 1468 = 367 × 4 (as stated in paper text)")
print(f"ACTUAL vs PAPER: {summ['augmented_count']} vs 1468  (factor {summ['factor']} vs 4)")
pd.DataFrame([summ]).to_csv(os.path.join(PROJECT, "results", "augmentation_summary.csv"), index=False)
print("\nsaved:", os.path.join(PROJECT, "results", "augmentation_summary.csv"))
print("manifest rows:", len(m))


SUMMARY (ACTUAL)
original_count      : 367
variants_per_image  : 22
augmented_count     : 8074
factor              : 22

PAPER (for comparison only): 1468 = 367 × 4 (as stated in paper text)
ACTUAL vs PAPER: 8074 vs 1468  (factor 22 vs 4)

saved: /content/drive/MyDrive/bsm_paper_replication/results/augmentation_summary.csv
manifest rows: 8074


## Figure: one original image with all its variants in a labelled grid (22 variants)

In [7]:
# ===== Figure: one original + all 22 variants =====
import os, random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
m = pd.read_csv("/content/augmentation_manifest.csv") if 'pd' not in locals() else pd.read_csv("/content/augmentation_manifest.csv")
import pandas as pd
m = pd.read_csv("/content/augmentation_manifest.csv")

# pick a random source that has all 22
srcs = m["source_image_id"].unique()
# just pick first for reproducibility with seed
s = srcs[0]
grp = m[m["source_image_id"]==s].sort_values("variant_name").copy()
print("source:", s, "| variants:", len(grp))
fig, axes = plt.subplots(5, 5, figsize=(20,20))
for i, (idx, row) in enumerate(grp.iterrows()):
    if i >= 25: break
    ax = axes.ravel()[i]
    try:
        with Image.open(row["filepath"]) as im:
            ax.imshow(im.convert("RGB"))
    except Exception:
        pass
    ax.set_title(row["variant_name"], fontsize=8)
    ax.axis("off")
for j in range(i+1, 25): axes.ravel()[j].axis("off")
fig.suptitle(f"Original+variants for {s}")
fig.tight_layout()
out = os.path.join(PROJECT, "figures", "nb02_one_source_all_variants.png")
fig.savefig(out, dpi=100)
plt.close()
print("saved:", out)


source: Ilmenite_1 | variants: 22
saved: /content/drive/MyDrive/bsm_paper_replication/figures/nb02_one_source_all_variants.png


## Persistence: save this notebook to the project folder

In [8]:
# ===== Save notebook copy to Drive =====
import glob, os, shutil, time
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT, "notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    dst = os.path.join(PROJECT, "notebooks", "02_data_augmentation.ipynb")
    shutil.copy(newest, dst)
    print("copied:", dst, "(modified", f"{(time.time()-os.path.getmtime(newest))/60:.1f}", "min ago)")
else:
    print("no autosave found yet")


no autosave found yet
